In [2]:
import os
import time
import shutil
from datetime import datetime
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive

from tqdm.notebook import trange
# from ipywidgets import IntProgress
# from IPython.display import display

## Functions

### Checkpoint functions

In [13]:
def create_ts(cp_file):
    with open(cp_file, 'w') as file:
        file.write(f'{datetime.now()}')

def save_ts(cp_file, dt):
    with open(cp_file, 'w') as file:
        file.write(f'{dt}')
    
def load_ts(cp_file):
    with open(cp_file, 'r') as file:
        return datetime.strptime(file.readline(), "%Y-%m-%d %H:%M:%S.%f")
    
def timeConvert(atime):
    dt = atime
    newtime = datetime.fromtimestamp(dt)                       
    return newtime

def sizeFormat(size):
    newform = format(size / 1024, '.2f')
    return newform + ' KB'

### Google drive functions

In [6]:
def get_drive():
    gauth = GoogleAuth(settings_file = "config/settings.yaml")
    gauth.LoadCredentialsFile("config/credentials.json")  # tenta carregar credenciais salvas

    if gauth.credentials is None:
        gauth.LocalWebserverAuth()  # primeira vez: faz login
    elif gauth.access_token_expired:
        gauth.Refresh()  # renova token expirado
    else:
        gauth.Authorize()  # já está autenticado

    gauth.SaveCredentialsFile("config/credentials.json")  # salva/atualiza token

    return GoogleDrive(gauth)

### Local functions

In [20]:
def move_meas(origin, destiny):
    shutil.move(origin, destiny)
    
def detect_new_meas(meas_path, ext, cp):
    # attrs = {
    #     'name': file_path.split('/')[-1],
    #     'size': sizeFormat(stats.st_size),
    #     'ctime': timeConvert(stats.st_ctime),
    #     'mtime': timeConvert(stats.st_mtime),
    #     'lmod': timeConvert(stats.st_atime),
    # }
    meas = {}
    for file in os.listdir(meas_path):
        if file[-len(ext):] == ext:
            file_ctime = timeConvert(os.stat(meas_path + file).st_ctime)
            if file_ctime > cp: # validate later: > or >=
                meas[file] = file_ctime
    
    # It is important to return sorted
    return dict(sorted(meas.items(), key = lambda item: item[1]))

def detect_new_meas_2(meas_path, ext):
    return [file for file in os.listdir(meas_path) if file[-len(ext):] == ext]

def send_meas(local_path, remote_path, meas, local_destiny_path, cp_file, verbose = False):
    
    drive = get_drive()
    
    if verbose:
        f = IntProgress(min = 0, max = len(meas), description = 'Progress:')
        display(f)

    # remember: 'meas' must be ordered
    
    for file_name, dt in meas.items():
        local_file = local_path + file_name

        remote_file = drive.CreateFile({
            'title': file_name,
            'parents': [{'id': remote_path}]
        })
        remote_file.SetContentFile(local_file)
        remote_file.Upload()
        
        move_meas(local_file, local_destiny_path)
        save_ts(cp_file, dt)        
        
        if verbose: 
            # print(f"File: {file_name} uploaded to Google Drive.")
            f.value += 1
    
def send_meas_2(local_path, remote_path, meas, local_backup_path):
    
    drive = get_drive()
    
    bar = trange(len(meas), desc = "Uploading")
    for i in bar:
        local_file = local_path + meas[i]

        remote_file = drive.CreateFile({
            'title': meas[i],
            'parents': [{'id': remote_path}]
        })
        remote_file.SetContentFile(local_file)
        remote_file.Upload()

        move_meas(local_file, local_backup_path)
        
    else:
        bar.refresh()   

## Main

#### old codes

In [29]:
# 
# if not os.path.exists(cp_file):
#     create_ts(cp_file)

# checkpoint = load_ts(cp_file)

# ------------------------------------

n = 5
temp_files = 'measurements/to_send/m'

for i in range(n):
    with open(temp_files + str(i) + '.txt', 'w') as file:
        file.write('empty')

In [28]:
conf_path = 'config/'
cp_file = conf_path + 'checkpoint.conf'
meas_to_send_path = 'measurements/to_send/'
meas_sent_path = 'measurements/sent/'
remote_meas_path = '18tk4XYY4z70DDTMJ_b5Nr7sTaZAqLyhE'
ext = 'txt'

meas = detect_new_meas_2(meas_to_send_path, ext)
send_meas_2(meas_to_send_path, remote_meas_path, meas, meas_sent_path)

Uploading:   0%|          | 0/5 [00:00<?, ?it/s]

In [8]:
print('teste', end = '\r')
print('olá')

oláte


olá


In [5]:
INTERVALO = 10
for restante in range(INTERVALO, 0, -1):
    print(f"⏳ Próxima execução em {restante} segundos...", end="\r")
    time.sleep(1)